# Active Learning with Uncertainty

Use model uncertainty to select informative training data.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.active import ActiveLearningLoop, UncertaintySampling
from deepuq.models import MLP
from deepuq.methods.mc_dropout import MCDropoutWrapper

## Setup

In [ ]:
# Generate 1D function with complex structure
np.random.seed(42)
torch.manual_seed(42)

def target_function(x):
    return np.sin(3*x) * np.exp(-0.3*x) + 0.5 * np.cos(5*x)

# Pool of unlabeled points
X_pool = np.linspace(0, 6, 200).reshape(-1, 1)
y_pool = target_function(X_pool) + np.random.randn(*X_pool.shape) * 0.05

# Initial training set: 5 random points
init_idx = np.random.choice(200, 5, replace=False)
X_init = X_pool[init_idx]
y_init = y_pool[init_idx]

print(f"Initial training points: {len(X_init)}")
print(f"Pool size: {len(X_pool)}")

plt.figure(figsize=(10, 4))
plt.plot(X_pool.ravel(), target_function(X_pool).ravel(), "k--", label="True function")
plt.scatter(X_init.ravel(), y_init.ravel(), c="red", s=100, zorder=5, label="Initial data")
plt.legend()
plt.title("Target Function and Initial Data")
plt.show()

## Run Active Learning Loop

In [ ]:
# Create model with MC Dropout for uncertainty
model = MLP(input_dim=1, hidden_dims=[64, 64], output_dim=1, p_drop=0.1)
uq_model = MCDropoutWrapper(model, n_mc=30, apply_softmax=False)

# Uncertainty sampling strategy - needs the model
strategy = UncertaintySampling(uq_model)

# Define a simple training function
def train_fn(model_wrapper, X_train, y_train):
    inner_model = model_wrapper.model
    opt = torch.optim.Adam(inner_model.parameters(), lr=1e-3)
    inner_model.train()
    for _ in range(100):
        opt.zero_grad()
        pred = inner_model(X_train)
        loss = torch.nn.functional.mse_loss(pred, y_train)
        loss.backward()
        opt.step()
    return model_wrapper

# Run active learning loop
pool_X = torch.tensor(X_pool, dtype=torch.float32)
pool_y = torch.tensor(y_pool, dtype=torch.float32)
init_X = torch.tensor(X_init, dtype=torch.float32)
init_y = torch.tensor(y_init, dtype=torch.float32)

loop = ActiveLearningLoop(
    model=uq_model,
    strategy=strategy,
    train_fn=train_fn,
    initial_X=init_X,
    initial_y=init_y,
    pool_X=pool_X,
    pool_y=pool_y,
)

history = loop.run(n_iterations=10, n_samples_per_iter=5)
print(f"Final training set size: {loop.train_X.shape[0]}")

## Visualize Acquisition

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# True function
X_dense = np.linspace(0, 6, 500).reshape(-1, 1)
ax.plot(X_dense.ravel(), target_function(X_dense).ravel(), "k--", linewidth=2, label="True function")

# Final model predictions
X_test_t = torch.tensor(X_dense, dtype=torch.float32)
result = uq_model.predict_uq(X_test_t)
mean = result.mean.numpy()
std = result.total_var.sqrt().numpy()
ax.fill_between(X_dense.ravel(), (mean - 2*std).ravel(), (mean + 2*std).ravel(),
                alpha=0.2, color="blue", label="\u00b1 2\u03c3")
ax.plot(X_dense.ravel(), mean.ravel(), "b-", label="Model mean")

# Initial points
ax.scatter(X_init.ravel(), y_init.ravel(), c="red", s=100, marker="^", zorder=6, label="Initial data")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Active Learning: Uncertainty-Guided Acquisition")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Compare Strategies

In [ ]:
# Show training sizes over iterations
train_sizes = [h['train_size'] for h in history]

plt.figure(figsize=(8, 5))
plt.plot(range(len(train_sizes)), train_sizes, "-o", label="Uncertainty Sampling")
plt.xlabel("Iteration")
plt.ylabel("Training Set Size")
plt.title("Active Learning: Training Set Growth")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()